# Beta Estimation Demo

This notebook compares TerraFin's beta estimates against the same yfinance-backed provider beta path the app already uses.

In [ ]:
import pandas as pd

from TerraFin import configure, load_terrafin_config
from TerraFin.analytics.analysis.risk import estimate_beta_5y_monthly, estimate_beta_5y_monthly_adjusted
from TerraFin.data.providers.market.ticker_info import get_ticker_info


configure()
config = load_terrafin_config()

## Compare TerraFin beta estimates with provider beta

In [ ]:
def provider_beta(symbol: str):
    info = get_ticker_info(symbol) or {}
    value = info.get("beta")
    return float(value) if value is not None else None


sample_tickers = ["NVDA", "AMZN", "KO", "005930.KS"]
rows = []
for symbol in sample_tickers:
    raw = estimate_beta_5y_monthly(symbol)
    adjusted = estimate_beta_5y_monthly_adjusted(symbol)
    provider = provider_beta(symbol)
    rows.append(
        {
            "symbol": symbol,
            "benchmark": raw.benchmark_symbol,
            "provider_beta": provider,
            "beta_5y_monthly": raw.beta,
            "beta_5y_monthly_adjusted": adjusted.beta,
            "abs_error_raw": None if provider is None or raw.beta is None else abs(raw.beta - provider),
            "abs_error_adjusted": None if provider is None or adjusted.beta is None else abs(adjusted.beta - provider),
            "observations": raw.observations,
            "r_squared": raw.r_squared,
            "status": raw.status,
            "warnings": "; ".join(raw.warnings),
        }
    )

comparison = pd.DataFrame(rows)
comparison


## Default reference method

`beta_5y_monthly` is TerraFin's current reference baseline. `beta_5y_monthly_adjusted` is included as a stability-oriented alternative.

In [ ]:
comparison[
    ["symbol", "provider_beta", "beta_5y_monthly", "beta_5y_monthly_adjusted", "abs_error_raw", "abs_error_adjusted"]
]


## Unsupported or unavailable cases

In [ ]:
unsupported = estimate_beta_5y_monthly("600519.SS")
pd.Series(
    {
        "symbol": unsupported.symbol,
        "method": unsupported.method_id,
        "status": unsupported.status,
        "benchmark": unsupported.benchmark_symbol,
        "warnings": "; ".join(unsupported.warnings),
    }
)
